**Domain – Media**
**focus –optimize the selection process**

##### Business challenge/requirement

Motion Studios is the largest Radio production house in Europe.  Their total revenue is $ 1B+. The companyhas launched a new reality show –"The Star RJ".  The show is about finding a new Radio Jockey who will be the star presenter onupcoming shows.
In the first round,participants have to upload their voice clip online and the clip will be evaluated by experts for selection inthe next round. There is a separate team inthefirst round for the evaluation of male and female voices.
Response to the show is unprecedented and the company is flooded with voice clips.
You as anML experthave to classify the voice as either male/female so that the first level of filtration is quicker.

**Key issues**
Voice samples are across accents

**Considerations**
The output from the pre-processed WAV files wassaved into the CSV file.

**Data volume**
- Approx 3000  records – file voice-classification.csv

Fields in Data

- meanfreq: mean frequency (in kHz)
- sd: standard deviation of the frequency•median: median frequency (in kHz)
- Q25: first quantile (in kHz)
- Q75: third quantile (in kHz)
- IQR: interquantile range (in kHz)
- skew: skewness (see note in specprop description)
- kurt: kurtosis (see note in specprop description)
- sp.ent: spectral entropy
- sfm: spectral flatness
- mode: mode frequency
- centroid: frequency centroid (see specprop)
- peakf: peak frequency (frequency with the highest energy)
- meanfun: average of fundamental frequency measured across the acoustic signal
- minfun: minimum fundamental frequency measured across the acoustic signal
- maxfun: maximum fundamental frequency measured across the acoustic signal
- meandom: average of dominant frequency measured across the acoustic signal
- mindom: minimum of dominant frequency measured across the acoustic signal
- maxdom: maximum of dominant frequency measured across the acoustic signal
- dfrange: range of dominant frequency measured across the acoustic signal
- modindx: modulation index. Calculated as the accumulated absolute difference between adjacent measurements of fundamental frequencies divided by the frequency range
- label: male or female

**Business benefits**

Since "The Star RJ" is a reality show, the time to select candidates is very short.  The whole success of the show and hence the profits depends upon quick and smooth execution

In [3]:
import pandas as pd
df = pd.read_csv('/content/voice-classification.csv')
df.head()

,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
0,0.059781,0.064241,0.032027,0.015071,0.090193,0.075122,12.863462,274.402906,0.893369,0.491918,...,0.059781,0.084279,0.015702,0.275862,0.007812,0.007812,0.007812,0.000000,0.000000,male
1,0.066009,0.067310,0.040229,0.019414,0.092666,0.073252,22.423285,634.613855,0.892193,0.513724,...,0.066009,0.107937,0.015826,0.250000,0.009014,0.007812,0.054688,0.046875,0.052632,male
2,0.077316,0.083829,0.036718,0.008701,0.131908,0.123207,30.757155,1024.927705,0.846389,0.478905,...,0.077316,0.098706,0.015656,0.271186,0.007990,0.007812,0.015625,0.007812,0.046512,male
3,0.151228,0.072111,0.158011,0.096582,0.207955,0.111374,1.232831,4.177296,0.963322,0.727232,...,0.151228,0.088965,0.017798,0.250000,0.201497,0.007812,0.562500,0.554688,0.247119,male
4,0.135120,0.079146,0.124656,0.078720,0.206045,0.127325,1.101174,4.333713,0.971955,0.783568,...,0.135120,0.106398,0.016931,0.266667,0.712812,0.007812,5.484375,5.476562,0.208274,male


In [4]:
#df.info()
pd.DataFrame(df.isnull().sum(), columns=['Count']).T

,meanfreq,sd,median,Q25,Q75,IQR,skew,kurt,sp.ent,sfm,...,centroid,meanfun,minfun,maxfun,meandom,mindom,maxdom,dfrange,modindx,label
Count,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA

pca = PCA(n_components=10)
lb = LabelEncoder()

df_reduced = pca.fit_transform(df.drop('label', axis=1))
df['label'] = lb.fit_transform(df['label'])

df_reduced = pd.DataFrame(df_reduced, columns=[f'PC{i+1}' for i in range(10)])


In [6]:
new_df = pd.concat([df_reduced, df['label']], axis=1)
new_df.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,label
0,238.081294,-4.802247,2.248039,-0.122248,0.123980,-0.202426,-0.026538,-0.132297,-0.081558,-0.147256,1
1,598.396593,-1.021374,0.881788,-0.021101,0.082521,-0.031257,-0.018231,-0.061454,-0.028554,-0.162663,1
2,988.762667,3.022605,-2.620214,0.047709,-0.122630,0.131213,-0.016569,-0.048415,-0.006888,-0.164207,1
3,-32.368181,-6.639116,-1.157241,-0.175888,0.156797,-0.006942,-0.007894,-0.010716,0.033846,0.007879,1
4,-32.286811,0.339167,-1.048950,-0.264576,0.341542,0.037312,0.008484,-0.009797,0.018290,-0.016661,1


In [7]:
from sklearn.model_selection import train_test_split
X = new_df.drop('label', axis=1)
y = new_df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [30]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

params = {
    'max_depth': [1,10, 20],
    'max_features': [10, 20],
    'criterion': ['gini', 'entropy'],
}

grid_search = GridSearchCV(estimator=DecisionTreeClassifier(random_state=42), param_grid=params, cv=3)

In [31]:
from sklearn.metrics import accuracy_score

grid_search.fit(X_train, y_train)
y_pred = grid_search.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best cross-validation score: {grid_search.best_score_}")
print(f"Test accuracy: {accuracy}")

Best parameters: {'criterion': 'entropy', 'max_depth': 10, 'max_features': 10}
Best cross-validation score: 0.8796372117371023
Test accuracy: 0.886435331230284
